In [1]:
# Install system packages
!apt-get update -y
!apt-get install -y python3-opengl ffmpeg xvfb swig cmake

# Install Python packages
!pip install gymnasium[box2d]
!pip install stable-baselines3[extra]
!pip install huggingface_sb3
!pip install huggingface_hub
!pip install pyvirtualdisplay

from IPython.display import clear_output
clear_output()
print("✅ All dependencies installed!")

✅ All dependencies installed!


In [2]:
# Colab has no screen — we need a virtual one to render videos
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

print("✅ Virtual display started!")

✅ Virtual display started!


In [3]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

print("✅ All imports successful!")

✅ All imports successful!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# ✅ Use v3 (v2 is deprecated)
env = make_vec_env("LunarLander-v3", n_envs=16)
# n_envs=16 means 16 parallel environments → trains much faster!

print("✅ Environment created:", env)

✅ Environment created: <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7beb5a9daae0>


In [8]:
model = PPO(
    policy="MlpPolicy",       # Multi-layer Perceptron
    env=env,
    n_steps=1024,             # Steps per environment per update
    batch_size=64,
    n_epochs=4,               # Number of training epochs per update
    gamma=0.999,              # Discount factor (high = values future rewards)
    gae_lambda=0.98,          # GAE lambda for advantage estimation
    ent_coef=0.01,            # Entropy bonus (encourages exploration)
    verbose=1                 # Print training progress
)

# Train for 1 million timesteps (~5-10 min on Colab)
model.learn(total_timesteps=1_000_000)

# Save the model locally
model.save("ppo-LunarLander-v3")
print("✅ Model trained and saved!")

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 91.4     |
|    ep_rew_mean     | -169     |
| time/              |          |
|    fps             | 3501     |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 16384    |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 91.1       |
|    ep_rew_mean          | -143       |
| time/                   |            |
|    fps                  | 2462       |
|    iterations           | 2          |
|    time_elapsed         | 13         |
|    total_timesteps      | 32768      |
| train/                  |            |
|    approx_kl            | 0.00690333 |
|    clip_fraction        | 0.0589     |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.38      |
|    explained_variance   | 0.00111    |
|    learning_rate        |

In [9]:
# Use v3 here too!
eval_env = Monitor(gym.make("LunarLander-v3"))

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"✅ Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")
# Target: mean_reward > 200 is considered "solved"

✅ Mean Reward: 238.56 +/- 50.86


In [10]:
# This opens a token input widget in Colab
notebook_login()

# OR use token directly (get from hf.co/settings/tokens)
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv

env_id = "LunarLander-v3"
model_name = "ppo-LunarLander-v3"
repo_id = "praveenkumar23/ppo-LunarLander-v3"  # ← Change this!

package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture="PPO",
    env_id=env_id,
    eval_env=DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")]),
    repo_id=repo_id,
    commit_message="Upload PPO LunarLander-v3 trained agent"
)

print(f"✅ Model uploaded! View at: https://huggingface.co/{repo_id}")

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saving video to /tmp/tmpej57t0sv/-step-0-to-step-1000.mp4


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Moviepy - Building video /tmp/tmpej57t0sv/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpej57t0sv/-step-0-to-step-1000.mp4



...

  ...-v3/pytorch_variables.pth: 100%|##########| 1.26kB / 1.26kB            

  ...r-v3/policy.optimizer.pth: 100%|##########| 88.7kB / 88.7kB            

  ...2x/ppo-LunarLander-v3.zip: 100%|##########|  150kB /  150kB            

  ...LunarLander-v3/policy.pth: 100%|##########| 44.1kB / 44.1kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/praveenkumar23/ppo-LunarLander-v3/tree/main/
✅ Model uploaded! View at: https://huggingface.co/praveenkumar23/ppo-LunarLander-v3


In [12]:
# Load from Hub
from huggingface_sb3 import load_from_hub
from stable_baselines3 import PPO

# Fix for pickle protocol mismatch (common error)
custom_objects = {
    "learning_rate": 0.0,
    "lr_schedule": lambda _: 0.0,
    "clip_range": lambda _: 0.0,
}

checkpoint = load_from_hub(
    repo_id="praveenkumar23/ppo-LunarLander-v3",
    filename="ppo-LunarLander-v3.zip"
)

model = PPO.load(checkpoint, custom_objects=custom_objects)
print("✅ Model loaded from Hub!")

ppo-LunarLander-v3.zip:   0%|          | 0.00/150k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


✅ Model loaded from Hub!


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [13]:
import os

# Record a video of the agent playing
eval_env = gym.make("LunarLander-v3", render_mode="rgb_array")
eval_env = RecordVideo(
    eval_env,
    video_folder="./videos",
    name_prefix="lunarlander",
    episode_trigger=lambda ep: True  # Record every episode
)

obs, _ = eval_env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    if terminated or truncated:
        obs, _ = eval_env.reset()

eval_env.close()

# Show video in Colab
from IPython.display import HTML
from base64 import b64encode

video_path = "./videos/" + os.listdir("./videos")[0]
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=600 controls><source src="{data_url}" type="video/mp4"></video>')

In [15]:
from huggingface_sb3 import package_to_hub
from stable_baselines3.common.vec_env import DummyVecEnv
import gymnasium as gym

package_to_hub(
    model=model,
    model_name="ppo-LunarLander-v3",
    model_architecture="PPO",
    env_id="LunarLander-v3",
    eval_env=DummyVecEnv([lambda: gym.make("LunarLander-v3", render_mode="rgb_array")]),
    repo_id="praveenkumar23/ppo-LunarLander-v3",  # ← change this!
    commit_message="Upload PPO LunarLander-v3 agent"
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saving video to /tmp/tmp2mosqdca/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmp2mosqdca/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmp2mosqdca/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmp2mosqdca/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo praveenkumar23/ppo-LunarLander-v3 to the Hugging Face
Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

  ...LunarLander-v3/policy.pth: 100%|##########| 44.1kB / 44.1kB            

  ...r-v3/policy.optimizer.pth: 100%|##########| 88.9kB / 88.9kB            

  ...j_/ppo-LunarLander-v3.zip: 100%|##########|  151kB /  151kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/praveenkumar23/ppo-LunarLander-v3/tree/main/


CommitInfo(commit_url='https://huggingface.co/praveenkumar23/ppo-LunarLander-v3/commit/9aaafc9e1d4218e50aa8635d26f29282f5088246', commit_message='Upload PPO LunarLander-v3 agent', commit_description='', oid='9aaafc9e1d4218e50aa8635d26f29282f5088246', pr_url=None, repo_url=RepoUrl('https://huggingface.co/praveenkumar23/ppo-LunarLander-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='praveenkumar23/ppo-LunarLander-v3'), pr_revision=None, pr_num=None)